# Deploy Contoso project into multi spoke

**Run this notebook once before the rest of the Foundry IQ multi-agent series.**

Deploys the Contoso multi-agent resources into the **existing** `rg-foundry-multi-{suffix}`
resource group, extending the shared AI Foundry account with a new `contoso-project` and
a Standard-SKU Azure AI Search service (`contoso-search-{suffix}`).

## What gets deployed

| Resource | Type | Where |
|----------|------|-------|
| `contoso-search-{suffix}` | Azure AI Search (Standard SKU) | New resource in `rg-foundry-multi-{suffix}` |
| `contoso-project` | Foundry Project | Child of existing `aif-spoke-multi-{suffix}` account - new |
| `contoso-apim-connection` | Project connection (ApiManagement) | On `contoso-project` - new |
| RBAC assignments | Deployer + project MI + search MI | On AI Account and Search service |

> **Standard SKU** is required for semantic search and `answerSynthesis` mode used by
> the Knowledge Bases in this lab. The existing `iq-search-{suffix}` (Foundry IQ) uses Basic
> SKU and is not upgraded - a dedicated search service provides clean isolation.

## Sequence

```
11-01-deploy-setup          ← this notebook (Bicep deployment, .env outputs)
11-02-index-and-ingest      ← create search indexes + upload Contoso sample data
11-03-knowledge-base-setup  ← create Knowledge Sources, KBs, MCP connections
11-04-multi-agent-setup     ← instantiate Agent Framework agents + validate routing
11-05-multi-agent-queries   ← WorkflowBuilder routing demos + citation display
```

## Prerequisites

1. **Multi-project spoke deployed** - `deploy-foundry-multi-project.ipynb` must have
   run successfully. The `.env` file must contain `MULTI_ACCOUNT`, `GATEWAY_URL`,
   `ALPHA_GATEWAY_KEY`, and `CHAT_MODEL`.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Step 1: Load configuration

In [1]:
import os
import json
import subprocess
import base64
import time
import tempfile
from pathlib import Path

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

# Read .env manually: same pattern as other deploy notebooks
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

GATEWAY_URL   = os.environ['GATEWAY_URL']
CHAT_MODEL    = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
MULTI_ACCOUNT = os.environ['MULTI_ACCOUNT']   # aif-spoke-multi-{suffix}, from the multi-project deployment

# Bootstrap key: used for the initial Bicep deployment.
# Step 5 creates a dedicated Contoso subscription and patches the connection.
BOOTSTRAP_KEY = os.environ['ALPHA_GATEWAY_KEY']

print(f'Gateway URL   : {GATEWAY_URL}')
print(f'Chat model    : {CHAT_MODEL}')
print(f'Multi account : {MULTI_ACCOUNT}')
print(f'Bootstrap key : {BOOTSTRAP_KEY[:4]}... (hidden)')

Gateway URL   : https://apim-foundry-c2676f.azure-api.net/openai
Chat model    : gpt-4.1-mini
Multi account : aif-spoke-multi-c2676f
Bootstrap key : 8382... (hidden)


## Step 2: Resolve resource group and principal

In [2]:
# Discover the resource group that contains the existing multi account
MULTI_RG = subprocess.run(
    f'az cognitiveservices account list --query "[?name==\'{MULTI_ACCOUNT}\'].resourceGroup" -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

assert MULTI_RG, f'Could not find resource group for account "{MULTI_ACCOUNT}" - is az login done?'

# Get subscription ID
SUB_ID = subprocess.run(
    'az account show --query id -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

# Get principal ID from cached JWT: avoids a network call to graph.microsoft.com
token   = subprocess.run(
    'az account get-access-token --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
padding = '=' * (4 - len(token.split('.')[1]) % 4)
PRINCIPAL_ID = json.loads(base64.b64decode(token.split('.')[1] + padding))['oid']

# Derive APIM service name from GATEWAY_URL for Step 5
# e.g. https://apim-foundry-{suffix}.azure-api.net/openai -> apim-foundry-{suffix}
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]
SUFFIX    = APIM_NAME.split('-')[-1]            # hub suffix (e.g. {suffix})
CORE_RG    = f'rg-foundry-core-{SUFFIX}'

print(f'Multi account : {MULTI_ACCOUNT}')
print(f'Multi RG      : {MULTI_RG}')
print(f'Subscription  : {SUB_ID}')
print(f'Principal ID  : {PRINCIPAL_ID}')
print(f'Core RG        : {CORE_RG}')
print(f'APIM service  : {APIM_NAME}')

Multi account : aif-spoke-multi-c2676f
Multi RG      : rg-foundry-multi-c2676f
Subscription  : 00000000-0000-0000-0000-000000000000
Principal ID  : 00000000-0000-0000-0000-000000000000
Core RG        : rg-foundry-core-c2676f
APIM service  : apim-foundry-c2676f


## Step 3: Deploy Contoso spoke Bicep

Deploys `main.bicep` into the existing `rg-foundry-multi-{suffix}` resource group.
Adds `contoso-project` and `contoso-search-{suffix}` to the shared account.
Takes ~3-5 minutes (Standard SKU search provisioning is slower than Basic).

In [3]:
result = subprocess.run(
    [
        'az', 'deployment', 'group', 'create',
        '-g', MULTI_RG,
        '--template-file', 'main.bicep',
        '-p', f'deployerPrincipalId={PRINCIPAL_ID}',
        '-p', f'apimUrl={GATEWAY_URL}',
        '-p', f'apimSubscriptionKey={BOOTSTRAP_KEY}',
        '-p', f'gatewayModelName={CHAT_MODEL}',
        '-p', f'existingAccountName={MULTI_ACCOUNT}',
        '-p', f'searchLocation=swedencentral',
        '--name', 'contoso-spoke',
        '-o', 'table',
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Bicep deployment failed - see stderr above')

Name           State      Timestamp                         Mode         ResourceGroup
-------------  ---------  --------------------------------  -----------  -----------------------
contoso-spoke  Succeeded  2026-05-13T08:52:05.081742+00:00  Incremental  rg-foundry-multi-c2676f



## Step 4: Read outputs and write to `.env`

In [4]:
r = subprocess.run(
    f'az deployment group show -g "{MULTI_RG}" -n contoso-spoke --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(f'Failed to read deployment outputs.\n{r.stderr}')

out = json.loads(r.stdout)

PROJECT_NAME     = out['projectName']['value']
PROJECT_ENDPOINT = out['projectEndpoint']['value']
PROJECT_MI       = out['projectManagedIdentityId']['value']
APIM_CONNECTION  = out['apimConnectionName']['value']
SEARCH_ENDPOINT  = out['searchEndpoint']['value']
SEARCH_NAME      = out['searchName']['value']

print(f'Account (existing) : {MULTI_ACCOUNT}')
print(f'Project name       : {PROJECT_NAME}')
print(f'Project endpoint   : {PROJECT_ENDPOINT}')
print(f'APIM connection    : {APIM_CONNECTION}')
print(f'Search endpoint    : {SEARCH_ENDPOINT}')
print(f'Search name        : {SEARCH_NAME}')

# Merge into .env
existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing.update({
    'CONTOSO_FOUNDRY_PROJECT':          PROJECT_NAME,
    'CONTOSO_FOUNDRY_PROJECT_ENDPOINT': PROJECT_ENDPOINT,
    'CONTOSO_APIM_CONNECTION':          APIM_CONNECTION,
    'CONTOSO_SEARCH_ENDPOINT':          SEARCH_ENDPOINT,
    'CONTOSO_SEARCH_NAME':              SEARCH_NAME,
    'CONTOSO_RESOURCE_GROUP':           MULTI_RG,
    'AZURE_SUBSCRIPTION_ID':            SUB_ID,
})

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'\n.env updated: {env_file}')

Account (existing) : aif-spoke-multi-c2676f
Project name       : contoso-project
Project endpoint   : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/contoso-project
APIM connection    : contoso-apim-connection
Search endpoint    : https://contoso-search-n5d3ja.search.windows.net
Search name        : contoso-search-n5d3ja

.env updated: <repo-root>/.env


## Step 5: Create dedicated Contoso APIM subscription

Creates a `foundry-gateway-contoso` subscription on the core APIM service, scoped to the
OpenAI API. This gives the Contoso multi-agent workload its own rate-limit bucket,
isolating its traffic from team inference quotas.

The dedicated key replaces the bootstrap key on the `contoso-apim-connection`.

> **If this step fails** (e.g. insufficient permissions on the core APIM), run the
> fallback cell to use `ALPHA_GATEWAY_KEY` instead.

In [ ]:
APIM_SUB_NAME = 'foundry-gateway-contoso'
APIM_BASE_URI = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
)

create_result = subprocess.run(
    [
        'az', 'rest', '--method', 'PUT',
        '--uri', f'{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}?api-version=2024-06-01-preview',
        '--body', json.dumps({
            'properties': {
                'displayName': 'Foundry Contoso Multi-Agent Gateway Access',
                'scope': (
                    f'/subscriptions/{SUB_ID}/resourceGroups/{CORE_RG}'
                    f'/providers/Microsoft.ApiManagement/service/{APIM_NAME}/apis/openai'
                ),
                'state': 'active',
            }
        }),
        '--headers', 'Content-Type=application/json',
    ],
    capture_output=True, text=True
)

if create_result.returncode != 0:
    print(f'APIM subscription creation failed:\n{create_result.stderr.strip()}')
    print('\nRun the fallback cell below to use ALPHA_GATEWAY_KEY instead.')
    CONTOSO_GATEWAY_KEY = None
else:
    CONTOSO_GATEWAY_KEY = subprocess.run(
        f'az rest --method POST'
        f' --uri "{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}/listSecrets?api-version=2024-06-01-preview"'
        f' --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    ).stdout.strip()

    # PATCH the contoso-apim-connection with the dedicated key.
    # isDefault must stay true: Foundry agent runtime resolves model_deployment_name
    # via the project's default APIM connection. PATCH replaces the properties block,
    # so the bicep-set flag must be re-asserted here.
    models_list = [{
        'name': CHAT_MODEL,
        'properties': {'model': {'name': CHAT_MODEL, 'version': '', 'format': 'OpenAI'}},
    }]
    connection_uri = (
        f'https://management.azure.com/subscriptions/{SUB_ID}'
        f'/resourceGroups/{MULTI_RG}/providers/Microsoft.CognitiveServices/accounts/{MULTI_ACCOUNT}'
        f'/projects/{PROJECT_NAME}/connections/{APIM_CONNECTION}?api-version=2025-04-01-preview'
    )
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump({
            'properties': {
                'category': 'ApiManagement',
                'target': GATEWAY_URL,
                'authType': 'ApiKey',
                'isDefault': True,
                'isSharedToAll': True,
                'credentials': {'key': CONTOSO_GATEWAY_KEY},
                'metadata': {
                    'deploymentInPath': 'true',
                    'inferenceAPIVersion': '2024-10-21',
                    'models': json.dumps(models_list),
                },
            }
        }, f)
        payload_file = f.name

    patch = subprocess.run(
        f'az rest --method PATCH --uri "{connection_uri}"'
        f' --body @"{payload_file}" --headers "Content-Type=application/json" -o none',
        shell=True, capture_output=True, text=True
    )
    if patch.returncode == 0:
        print(f'APIM subscription created : {APIM_SUB_NAME}')
        print(f'Contoso gateway key       : {CONTOSO_GATEWAY_KEY[:4]}... (hidden)')
        print(f'Connection patched        : {APIM_CONNECTION} on {PROJECT_NAME}')
    else:
        print(f'Connection PATCH failed: {patch.stderr.strip()}')
        CONTOSO_GATEWAY_KEY = None

In [6]:
# Fallback: use ALPHA_GATEWAY_KEY if Step 5 failed.
# Uncomment and run only if needed.

# CONTOSO_GATEWAY_KEY = os.environ['ALPHA_GATEWAY_KEY']
# print(f'Using ALPHA_GATEWAY_KEY as CONTOSO_GATEWAY_KEY: {CONTOSO_GATEWAY_KEY[:4]}... (hidden)')

In [7]:
assert CONTOSO_GATEWAY_KEY, 'CONTOSO_GATEWAY_KEY is not set - check Step 5 or run the fallback cell.'

existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing['CONTOSO_GATEWAY_KEY'] = CONTOSO_GATEWAY_KEY
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'CONTOSO_GATEWAY_KEY written to {env_file}')

CONTOSO_GATEWAY_KEY written to <repo-root>/.env


## Step 6: Wait for RBAC propagation

Azure role assignments can take up to 2 minutes to propagate. Running the subsequent
notebooks immediately may produce 403 errors on the project or search service.

In [8]:
from IPython.display import clear_output

for remaining in range(90, 0, -10):
    clear_output(wait=True)
    print(f'Waiting for RBAC to propagate... {remaining}s')
    time.sleep(10)

clear_output(wait=True)
print('RBAC propagation wait complete.')

RBAC propagation wait complete.


## Step 7: Verify deployment

In [9]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.search.documents.indexes import SearchIndexClient

credential = DefaultAzureCredential()

# Verify AI Search service is reachable
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
indexes = list(index_client.list_index_names())
print(f'Search endpoint  : {SEARCH_ENDPOINT}')
print(f'Existing indexes : {indexes or "(none - expected on first deploy)"}')

# Verify Foundry project
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
connections = list(project_client.connections.list())
print(f'\nProject          : {PROJECT_ENDPOINT}')
print(f'Connections:')
for c in connections:
    d = dict(c)
    print(f'  {d.get("name")} ({d.get("type")}) -> {d.get("target")}')

Search endpoint  : https://contoso-search-n5d3ja.search.windows.net
Existing indexes : (none — expected on first deploy)

Project          : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/contoso-project
Connections:
  appinsights-connection (AppInsights) -> /subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/rg-foundry-multi-c2676f/providers/Microsoft.Insights/components/appi-obs-n5d3ja
  contoso-apim-connection (ApiManagement) -> https://apim-foundry-c2676f.azure-api.net/openai


## Done

The Contoso resources are deployed and `.env` has been updated.

**Keys written to `.env`:**

| Key | Description |
|-----|-------------|
| `CONTOSO_FOUNDRY_PROJECT` | Project name (`contoso-project`) |
| `CONTOSO_FOUNDRY_PROJECT_ENDPOINT` | Project endpoint URL |
| `CONTOSO_APIM_CONNECTION` | APIM connection name on `contoso-project` |
| `CONTOSO_SEARCH_ENDPOINT` | Azure AI Search endpoint URL |
| `CONTOSO_SEARCH_NAME` | Azure AI Search service name |
| `CONTOSO_GATEWAY_KEY` | Dedicated APIM subscription key for Contoso workload |
| `CONTOSO_RESOURCE_GROUP` | Resource group (same as `rg-foundry-multi-{suffix}`) |
| `AZURE_SUBSCRIPTION_ID` | Azure subscription ID |

**Next step:** run `11-02-index-and-ingest.ipynb` to create search indexes and upload Contoso sample data.